**this is for running all 4 models (BART, BERT, MPNet, Jina Embeddings V2)**
- doing this before finishing the actual module, so we have data for the extended abstract
- testing on all 100 data inputs, with both the original and 3-word codes
- sections: 0) data setup, 1) BART, 2) BERT, 3) MPNet, 4) Jina Embeddings V2, 5) getting eval results text file
- with subsections 1) getting all the scores in text files, 2) evaluation

**0) Data Setup**

In [1]:
raw_data = open("Pain points data-ground truth.txt", "r") 
data = raw_data.read() 

data_list = data.split("\n")
for i in range(len(data_list)):
    d = data_list[i].split("\t")
    data_list[i] = d

#remove extra characters
for string in data_list:
    if "\n" in string:
        new_string = string.replace("\n", "")
        #print(string)
        i = data_list.index(string)
        data_list[i] = new_string
    if '"' in string:
        new_string = string.replace('"', "")
        i = data_list.index(string)
        data_list[i] = new_string

print(data_list) 
raw_data.close() 

testable_data = []
ground_truths = []
for p in range(len(data_list)):
    testable_data.append(data_list[p][0])
    ground_truths.append(data_list[p][1])

print(len(testable_data)) #for reference
print(testable_data)
print(len(ground_truths))
print(ground_truths)

og_codes = ["Lathe chuck overtightend", "18 V Drills too large to use with one hand", "Disposable gloves are a large size", "Miter saw is on tall table (awkward to use)", "People forget to put away the clamps", "Someone left sawdust and wood chips everywhere", "Someone squeezed through aisle and bumped user (bumped e-stop)", "Wood scraps too small to be useful", "Digging through the unlabeled cabinets looking for drill", "Trash bag is ripped", "Had to wait for epoxy to cure"]
print(og_codes) #for reference
short_codes = ["Lathe chuck overtightened", "Drill too large", "Gloves size large", "Tall miter saw", "Clamps scattered around", "Sawdust, wood everywhere", "Bumped into user", "Useless wood scraps", "Unlabeled drill cabinet", "Ripped trash bag", "Epoxy wait time"]
print(short_codes) #for reference

[['cutting wood', '0'], ['didn’t know how to use lathe', '1'], ['Finding drill', '9'], ['Taking out trash', '10'], ['Finding clamp', '5'], ['Loosening the lathe', '1'], ['Cleaning up after others', '6'], ['Equipment too big', '2'], ['Too small of wood scraps.', '8'], ['Had to take trash out.', '10'], ['Had to find a clamp.', '5'], ['Had to dig in unlabeled container for a 12v drill.', '9'], ['Got bumped while using sander, accidentally pressed emergency stop button.', '7'], ['Had to clean off messy workspace.', '6'], ['Gloves were too big.', '3'], ['People putting garbage in the scrp bin', '8'], ['People not throwing trash properly ripping the bag', '10'], ['People not putting things where they go', '5'], ['People not putting things back the way they were found', '5'], ['People leaving a mess and not cleaning up', '6'], ['bumped by person using belt sander', '7'], ['garbage bag had hole', '10'], ['sawdust left on machine', '6'], ['lathe chuck tightened too tight', '1'], ['only found 18

In [2]:
#process into a dataset 
from datasets import Dataset
from transformers.pipelines.pt_utils import KeyDataset

data_numbers = []
for i in range(len(testable_data)):
     data_numbers.append(i)

data = {
    'TD': data_numbers,
    'text': testable_data
}
dataset = Dataset.from_dict(data)
print(dataset)

Dataset({
    features: ['TD', 'text'],
    num_rows: 103
})


**1.1) BART scoring**

In [ ]:
#model set-up
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

import sys

original_stdout = sys.stdout

with open('output-long-BART.txt', 'w') as f:
    sys.stdout = f

    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        #sequence_to_classify = testable_data[i]
        sequence_to_classify = KeyDataset(dataset, "text")[i]
        for code in og_codes:
            candidate_labels = code
            results = classifier(sequence_to_classify, candidate_labels, multi_label=False) #should I keep multi_labels=True? (whats the diff)
            print(str(results["scores"][0]) + "\t" + code) #only gimme the scores

f.close()

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
#model set-up
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

import sys

original_stdout = sys.stdout

with open('output-short-BART.txt', 'w') as f:
    sys.stdout = f

    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        #sequence_to_classify = testable_data[i]
        sequence_to_classify = KeyDataset(dataset, "text")[i]
        for code in short_codes:
            candidate_labels = code
            results = classifier(sequence_to_classify, candidate_labels, multi_label=False) #should I keep multi_labels=True? (whats the diff)
            print(str(results["scores"][0]) + "\t" + code) #only gimme the scores

f.close()

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
